In [1]:
from execute_util import text, link, image
from facts import a100_flop_per_sec, h100_flop_per_sec
import torch.nn.functional as F
import timeit
import torch
from typing import Iterable
from torch import nn
import numpy as np
from lecture_util import article_link
from jaxtyping import Float
from einops import rearrange, einsum, reduce
from references import zero_2019


In [2]:
text("You can create tensors in multiple ways:")
x = torch.tensor([[1., 2, 3], [4, 5, 6]])  # @inspect x
x = torch.zeros(4, 8)  # 4x8 matrix of all zeros @inspect x
x = torch.ones(4, 8)  # 4x8 matrix of all ones @inspect x
x = torch.randn(4, 8)  # 4x8 matrix of iid Normal(0, 1) samples @inspect x

text("Allocate but don't initialize the values:")
x = torch.empty(4, 8)  # 4x8 matrix of uninitialized values @inspect x
text("...because you want to use some custom logic to set the values later")
nn.init.trunc_normal_(x, mean=0, std=1, a=-2, b=2)  # @inspect x

tensor([[-0.7516, -1.2593, -0.9827, -0.3874,  1.2032, -0.9820,  0.8552, -0.4620],
        [-0.1127, -0.3373, -1.2762, -1.2981,  0.9977, -1.9944, -0.3801, -0.3943],
        [-1.0611,  0.1952, -0.1179,  0.8171,  1.2489, -1.5675,  0.4076,  1.1857],
        [-0.0630, -0.0586,  0.6618,  0.3541,  1.0032, -0.2078, -1.6866,  1.2515]])

In [4]:
text("Let's examine memory usage of these tensors.")
text("Memory is determined by the (i) number of values and (ii) data type of each value.")
x = torch.zeros(4, 8)  # @inspect x
def get_memory_usage(tensor: torch.Tensor) -> int:
    """Calculate memory usage in bytes for a given tensor."""
    return tensor.numel() * tensor.element_size()
assert x.dtype == torch.float32  # Default type
assert x.numel() == 4 * 8
assert x.element_size() == 4  # Float is 4 bytes
assert get_memory_usage(x) == 4 * 8 * 4  # 128 bytes

In [5]:
x = torch.tensor([
    [0., 1, 2, 3],
    [4, 5, 6, 7],
    [8, 9, 10, 11],
    [12, 13, 14, 15],
])

text("To go to the next row (dim 0), skip 4 elements in storage.")
assert x.stride(0) == 4

text("To go to the next column (dim 1), skip 1 element in storage.")
assert x.stride(1) == 1

text("To find an element:")
r, c = 1, 2
index = r * x.stride(0) + c * x.stride(1)  # @inspect index
assert index == 6

In [6]:
def same_storage(x: torch.Tensor, y: torch.Tensor):
    return x.untyped_storage().data_ptr() == y.untyped_storage().data_ptr()

x = torch.tensor([[1., 2, 3], [4, 5, 6]])  # @inspect x

text("Many operations simply provide a different **view** of the tensor.")
text("This does not make a copy, and therefore mutations in one tensor affects the other.")

text("Get row 0:")
y = x[0]  # @inspect y
assert torch.equal(y, torch.tensor([1., 2, 3]))
assert same_storage(x, y)

In [8]:
text("Get column 1:")
y = x[:, 1]  # @inspect y
assert torch.equal(y, torch.tensor([2, 5]))
assert same_storage(x, y)

In [9]:
text("View 2x3 matrix as 3x2 matrix:")
y = x.view(3, 2)  # @inspect y
assert torch.equal(y, torch.tensor([[1, 2], [3, 4], [5, 6]]))
assert same_storage(x, y)

In [10]:
text("Transpose the matrix:")
y = x.transpose(1, 0)  # @inspect y
assert torch.equal(y, torch.tensor([[1, 4], [2, 5], [3, 6]]))
assert same_storage(x, y)

In [11]:
text("Check that mutating x also mutates y.")
x[0][0] = 100  # @inspect x, @inspect y
assert y[0][0] == 100